In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical

In [ ]:
data = fetch_openml(name='letter', version=1, as_frame=True)
X = data.data.astype(float)
y = data.target
print('Shape:', X.shape)
print('Classes:', sorted(y.unique()))

In [ ]:
le = LabelEncoder()
y_enc = le.fit_transform(y)
y_cat = to_categorical(y_enc, 26)

X_train, X_test, y_train, y_test, y_tr_e, y_te_e = train_test_split(
    X, y_cat, y_enc, test_size=0.2, random_state=42)

sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test  = sc.transform(X_test)

In [ ]:
model = Sequential([
    Dense(256, activation='relu', input_dim=16),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(26, activation='softmax')
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
history = model.fit(X_train, y_train, epochs=30, batch_size=64, validation_split=0.1, verbose=1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy'); axes[0].legend()
axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('Loss'); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
loss, acc = model.evaluate(X_test, y_test)
print(f'Test Accuracy: {acc*100:.2f}%')

y_pred = np.argmax(model.predict(X_test), axis=1)
print(classification_report(y_te_e, y_pred, target_names=le.classes_))

In [ ]:
cm = confusion_matrix(y_te_e, y_pred)
plt.figure(figsize=(16, 13))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.tight_layout(); plt.show()